# Question 7

In [0]:
import pandas as pd

data = [
    {"order_id": 1, "customer": "Vikas", "amount": 500, "city": "Jaipur"},
    {"order_id": 2, "customer": "Priya", "amount": 700, "city": "Delhi"},
    {"order_id": 3, "customer": "Rahul", "amount": 300, "city": "Jaipur"},
    {"order_id": 4, "customer": "Amit", "amount": 900, "city": "Delhi"},
    {"order_id": 5, "customer": "Neha", "amount": 600, "city": "Mumbai"},
    {"order_id": 6, "customer": "Ravi", "amount": 400, "city": "Mumbai"},
]

df = pd.DataFrame(data)

result = (
    df[df["amount"] > 500]
    .groupby("city")["amount"]
    .sum()
    .reset_index()
)

print(result)

In [0]:
from pyspark.sql.functions import sum

data = [
    {"order_id": 1, "customer": "Vikas", "amount": 500, "city": "Jaipur"},
    {"order_id": 2, "customer": "Priya", "amount": 700, "city": "Delhi"},
    {"order_id": 3, "customer": "Rahul", "amount": 300, "city": "Jaipur"},
    {"order_id": 4, "customer": "Amit", "amount": 900, "city": "Delhi"},
    {"order_id": 5, "customer": "Neha", "amount": 600, "city": "Mumbai"},
    {"order_id": 6, "customer": "Ravi", "amount": 400, "city": "Mumbai"},
]


df = spark.createDataFrame(data)
new_df =  df.filter('amount > 500').groupBy('city').agg(
    sum('amount').alias('amount')
)

new_df.display()



| Pandas                | Problem at large scale                                               | PySpark solution                                                                             |
| --------------------- | -------------------------------------------------------------------- | -------------------------------------------------------------------------------------------- |
| `pd.read_csv()`       | Large file ko generally single machine ki memory mein load karta hai | Spark data ko partitions mein divide karke multiple machines/cores par process kar sakta hai |
| `groupby()`           | Huge dataset par memory/processing bottleneck aa sakta hai           | Spark distributed `groupBy()` use karta hai                                                  |
| Large transformations | Pandas primarily single-machine processing hai                       | Spark distributed execution + parallel processing use karta hai                              |


# Question 8

In [0]:
from pyspark.sql.functions import *



df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/cyntexa_dev/sales/my_volume2/*.csv")
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("event_timestamp", current_timestamp())
)



# 2. Extract partition columns (e.g., year and month from a date/timestamp field)
df_with_partitions = df \
    .withColumn("year", year(col("event_timestamp"))) \
    .withColumn("month", month(col("event_timestamp")))


# 3. Write data using partitionBy

df_with_partitions.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("year", "month") \
    .option("targetFileSize", "128MB") \
    .saveAsTable("cyntexa_dev.sales.sales_table_partition")





# Partitioning Strategy: 
Since table mostly queried using data ranges, partitioning the data by the year , month is appropriate. This organizes the data into separate partitions such as year=2026/month=8 . When a quesry fires on these partition columns, spark can use partition pruning and avoid irrelevant records   

# File Size Strategy : 

A target file size of approximately 128 MB is used to maintain a balance between too many small files and large files .

# Lazy Evaluation : 

spark transformations such withColumn, year(), month()  are lazy , they do not immediately execute  the computition. Instead spark builds logical execution plan , and actual computation begins when an action such as saveAsTable()  ,count()  is executed.

# Physical Plan:
When the action is triggered, Spark converts the logical plan into an optimized physical plan and executes it. During query execution, Spark can apply optimizations such as partition pruning, allowing it to read only the relevant year/month partitions instead of scanning the entire table.



In [0]:
%sql
select * from cyntexa_dev.sales.sales_table_partition

# Question 9
- (Data Analyst) Using a Spark DataFrame (not SQL), reproduce a report you'd normally build in Excel/pandas — e.g., monthly revenue by category — and export the result for a dashboard. 

In [0]:
from pyspark.sql.window import * 
from pyspark.sql.functions import * 

df_customer = spark.read.table('samples.tpch.customer')
df_nation = spark.read.table('samples.tpch.nation')
df_orders = spark.read.table('samples.tpch.orders')
df_region = spark.read.table('samples.tpch.region')

df = df_customer.join(df_nation, df_customer['c_nationkey'] == df_nation['n_nationkey']).join(df_orders, df_customer['c_custkey'] == df_orders['o_custkey']).join(df_region, df_nation['n_regionkey'] == df_region['r_regionkey']).filter(df_orders['o_orderstatus'] == 'F')

customer_wise_region_amount =  df.select('c_custkey', 'c_name', 'o_orderstatus', 'o_totalprice' , "r_regionkey",'r_name')

# Customer-wise total within region
region_wise_total = (
    customer_wise_region_amount
    .groupBy(
        'r_regionkey',
        'r_name',
        'c_custkey',
        'c_name'
    )
    .agg(
        sum('o_totalprice').alias('region_wise_total')
    )
)

# Rank customers within each region
wind = Window.partitionBy("r_regionkey").orderBy(
    col("region_wise_total").desc()
)

region_wise_total = (
    region_wise_total
    .withColumn("rnk", rank().over(wind))
    .filter(col("rnk") <= 5)
)




region_wise_total.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("cyntexa_dev.analytics.region_wise_top5_customers")



> - Report: Region-wise Top 5 Customers

This report identifies the top 5 customers in each region based on their
total spending from completed orders. The report is generated using Spark
DataFrame transformations, aggregation, and window functions without using SQL.
The final result is stored in the Gold layer for dashboard consumption.

In [0]:
spark.sql('''
          select * from cyntexa_dev.analytics.region_wise_top5_customers
 ''')